In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
import os

# Set constants
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 120
# Define paths (adjust as needed)
base_path = "/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset"
train_dir = os.path.join(base_path, 'Training')
test_dir = os.path.join(base_path, 'Testing')

# Data Augmentation
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=15,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

# Generators
train_generator = train_datagen.flow_from_directory(train_dir,
                                                    target_size=(IMG_SIZE, IMG_SIZE),
                                                    batch_size=BATCH_SIZE,
                                                    class_mode='categorical')

test_generator = test_datagen.flow_from_directory(test_dir,
                                                  target_size=(IMG_SIZE, IMG_SIZE),
                                                  batch_size=BATCH_SIZE,
                                                  class_mode='categorical')

from tensorflow.keras import layers, models, regularizers, callbacks

model2 = models.Sequential()

# Input Layer
model2.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                         kernel_regularizer=regularizers.l2(0.001),
                         input_shape=(IMG_SIZE, IMG_SIZE, 3)))

model2.add(layers.BatchNormalization())
model2.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Dropout(0.25))

# 64 filters
model2.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model2.add(layers.BatchNormalization())
model2.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Dropout(0.3))

# 128 filters
model2.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
model2.add(layers.BatchNormalization())
model2.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Dropout(0.3))

# 256 filters
model2.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
model2.add(layers.BatchNormalization())
model2.add(layers.Conv2D(256, (3, 3), activation='relu', padding='same'))
model2.add(layers.MaxPooling2D((2, 2)))
model2.add(layers.Dropout(0.35))

# Global Pooling + Dense
model2.add(layers.GlobalAveragePooling2D())
model2.add(layers.Dense(256, activation='relu'))
model2.add(layers.Dropout(0.4))
model2.add(layers.Dense(4, activation='softmax'))  # 4 classes

model2.summary()

# Compile the model
model2.compile(optimizer=Adam(learning_rate=0.0001),
               loss='categorical_crossentropy',
               metrics=['accuracy'])


# Train the model
history = model2.fit(train_generator,
                     validation_data=test_generator,
                     epochs=EPOCHS)

# Save the model
model2.save("improved_brain_tumor_classifier.keras")

2025-05-17 08:14:51.886481: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747469692.075917      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747469692.139424      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1747469711.172915      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 224, 224, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 224, 224, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 112, 112, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 112, 112, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 56, 56, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 56, 56, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_6 (Conv2D)                    │ (None, 28, 28, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 28, 28, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_7 (Conv2D)                    │ (None, 28, 28, 256)         │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 14, 14, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

...

Epoch 1/120


I0000 00:00:1747469720.831477      73 service.cc:148] XLA service 0x7845bc019ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1747469720.832723      73 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1747469721.746191      73 cuda_dnn.cc:529] Loaded cuDNN version 90300


  2/179 ━━━━━━━━━━━━━━━━━━━━ 13s 78ms/step - accuracy: 0.3125 - loss: 1.6489   

I0000 00:00:1747469734.486358      73 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


179/179 ━━━━━━━━━━━━━━━━━━━━ 136s 647ms/step - accuracy: 0.5745 - loss: 1.0003 - val_accuracy: 0.2304 - val_loss: 1.7025
Epoch 2/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 70s 389ms/step - accuracy: 0.7139 - loss: 0.7167 - val_accuracy: 0.2288 - val_loss: 1.8770
Epoch 3/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 72s 403ms/step - accuracy: 0.7656 - loss: 0.6110 - val_accuracy: 0.3760 - val_loss: 3.7579
Epoch 4/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 79s 441ms/step - accuracy: 0.8051 - loss: 0.5274 - val_accuracy: 0.4050 - val_loss: 4.4686
Epoch 5/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 88s 491ms/step - accuracy: 0.8347 - loss: 0.4604 - val_accuracy: 0.3783 - val_loss: 2.9827
Epoch 6/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 76s 424ms/step - accuracy: 0.8479 - loss: 0.4167 - val_accuracy: 0.3883 - val_loss: 2.7593
Epoch 7/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 76s 423ms/step - accuracy: 0.8572 - loss: 0.3974 - val_accuracy: 0.4302 - val_loss: 2.7139
Epoch 8/120
179/179 ━━━━━━━━━━━━━━━━━━━━ 76s 425ms/step - accuracy: 0.8687 - loss: 0.36

In [2]:
import numpy as np
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Load the saved model
model = tf.keras.models.load_model("improved_brain_tumor_classifier.keras")

# Define class labels (based on your dataset structure)
class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']

# Load and preprocess the image
img_path = '/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset/Testing/glioma/Te-glTr_0000.jpg'  # Replace with actual path

for i in range(10,30):
    img_path = f'/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset/Testing/glioma/Te-gl_00{i}.jpg'  # Replace with actual path
    
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    
    # Predict
    pred = model.predict(img_array)
    predicted_class = class_names[np.argmax(pred)]
    print(predicted_class)
# Display

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
glioma
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
glioma


In [3]:
img_path = '/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset/Testing/meningioma/Te-glTr_0000.jpg'  # Replace with actual path

for i in range(10,30):
    img_path = f'/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset/Testing/pituitary/Te-pi_00{i}.jpg'  # Replace with actual path
    
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    
    # Predict
    pred = model.predict(img_array)
    predicted_class = class_names[np.argmax(pred)]
    print(predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
pituitary
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
pituitary
